<a href="https://colab.research.google.com/github/rupanjali-bharti/Twitter-Data-Sentiment-Analysis/blob/main/twitter_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving twitter_training.csv to twitter_training.csv


In [2]:
!pip install torch torchvision torchaudio transformers scikit-learn pandas
import pandas as pd
import torch
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW  # <--- Changed this
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

# Load Dataset (Specify the correct path to your Kaggle download)
# The dataset has no headers, so we name them manually.
cols = ['id', 'entity', 'sentiment', 'text']
df = pd.read_csv('twitter_training.csv', names=cols)

# 1. Preprocessing: Drop 'Irrelevant' or treat as 'Neutral'
# BERT handles 3-class classification well.
df = df[df['sentiment'] != 'Irrelevant']
df.dropna(subset=['text'], inplace=True)

# Map labels to integers
label_map = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
df['label'] = df['sentiment'].map(label_map)

# Use a sample for faster training if you don't have a high-end GPU
df = df.sample(5000, random_state=42)

In [3]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset, RandomSampler, SequentialSampler
from tqdm import tqdm

# 1. Load Tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def encode_tweets(tweets, labels):
    # Ensure all tweets are strings and remove any NaN values
    tweets = tweets.fillna("").astype(str).tolist()

    # CALLING THE TOKENIZER DIRECTLY (Modern replacement for encode_plus)
    encoded_batch = tokenizer(
        tweets,
        add_special_tokens=True,
        max_length=64,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    return encoded_batch['input_ids'], encoded_batch['attention_mask'], torch.tensor(labels.values)

# 2. Prepare Data
# Assuming 'df' is already loaded from the previous steps
X_train, X_val, y_train, y_val = train_test_split(df['text'], df['label'], test_size=0.2)

print("Encoding started...")
train_inputs, train_masks, train_labels = encode_tweets(X_train, y_train)
val_inputs, val_masks, val_labels = encode_tweets(X_val, y_val)

# 3. Create DataLoaders
train_loader = DataLoader(
    TensorDataset(train_inputs, train_masks, train_labels),
    sampler=RandomSampler(train_inputs),
    batch_size=32
)

val_loader = DataLoader(
    TensorDataset(val_inputs, val_masks, val_labels),
    sampler=SequentialSampler(val_inputs),
    batch_size=32
)

print(f"Success! Training batches: {len(train_loader)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Encoding started...
Success! Training batches: 125


In [4]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3

for epoch in range(epochs):
    model.train()
    for batch in tqdm(train_loader):
        b_input_ids, b_input_mask, b_labels = tuple(t.to(device) for t in batch)
        model.zero_grad()
        outputs = model(b_input_ids, attention_mask=b_input_mask, labels=b_labels)
        outputs.loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} complete.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
 42%|████▏     | 52/125 [18:55<26:33,

KeyboardInterrupt: 

In [1]:
model.eval() # Set model to evaluation mode
val_preds, val_true = [], []

print("Running Evaluation...")
for batch in tqdm(val_loader):
    b_input_ids, b_input_mask, b_labels = tuple(t.to(device) for t in batch)

    with torch.no_grad(): # Disable gradient calculation (saves memory)
        outputs = model(b_input_ids, attention_mask=b_input_mask)

    logits = outputs.logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    val_preds.extend(np.argmax(logits, axis=1).flatten())
    val_true.extend(label_ids.flatten())

# Print Classification Report
from sklearn.metrics import classification_report
print(classification_report(val_true, val_preds, target_names=['Negative', 'Neutral', 'Positive']))

NameError: name 'model' is not defined

In [2]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64).to(device)
    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=1).item()
    labels = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    return labels[prediction]

# Example Test
test_tweet = "I am absolutely hating this new update, it works so slowly!"
print(f"Result: {predict_sentiment(test_tweet)}")

NameError: name 'tokenizer' is not defined

In [3]:
# Saves the model and the tokenizer to a folder
model.save_pretrained('./bert_sentiment_model')
tokenizer.save_pretrained('./bert_sentiment_model')
print("Model saved to disk.")

NameError: name 'model' is not defined

In [5]:
!pip install gradio

In [ ]:
import gradio as gr

def sentiment_ui_wrapper(text):
    # This uses the predict_sentiment function we created earlier
    try:
        result = predict_sentiment(text)
        return f"Predicted Sentiment: {result}"
    except Exception as e:
        return f"Error: Ensure your model is trained and loaded. Details: {str(e)}"

# Create the Gradio Interface
description = """
### BERT Sentiment Analyzer
Enter a paragraph below to see if the model detects it as **Positive**, **Neutral**, or **Negative**.
"""

interface = gr.Interface(
    fn=sentiment_ui_wrapper,
    inputs=gr.Textbox(lines=5, placeholder="Enter a paragraph here...", label="Input Text"),
    outputs=gr.Textbox(label="Model Prediction"),
    title="Twitter Sentiment Analysis UI",
    description=description,
    theme="soft"
)

# launch(share=True) gives you a public URL (gradio.live) valid for 72 hours
interface.launch(share=True)